In [2]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import utils
import statsmodels.formula.api as smf


audio_paths = [
    f"../music-clips/classical_{i}.wav" for i in range(1,17) ] + [
    f"../music-clips/electronic_{i}.wav" for i in range(1,17)
]
DATA_DIR = Path("../subject-data")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mad_dict = utils.load_pickle("mad_data.pkl")
mad_df = utils.mad_dict_to_dataframe(mad_dict)

In [6]:
pickle_files = sorted(list(DATA_DIR.glob("*.pickle")) + list(DATA_DIR.glob("*.pkl")))
print(f"Found {len(pickle_files)} pickle files")
timeseries_rows = []
trial_rows = []

for pf in pickle_files:
    print("Loading:", pf.name)

    with open(pf, "rb") as f:
        outVars = pickle.load(f)
        print(outVars)

    expInfo = outVars["expInfo"]
    trialInfo = outVars["trialInfo"]

    subject_id = expInfo["participant_number"]
    n_trials = len(trialInfo["trial"])

    for t in range(n_trials):
        trial_number = trialInfo["trial"][t]
        piece_id = trialInfo["musFile"][t]
        genre = trialInfo["musGenre"][t]
        overall_rating = trialInfo["overall_rating_value"][t]

        dial_vals = trialInfo["dial_values"][t][:3601]
        dial_times = trialInfo["dial_times"][t][:3601]

        if len(dial_vals) != 3601 or len(dial_times) != 3601:
            raise ValueError(f"Trial {trial_number} in {pf.name} does not have 3601 samples.")

        mad_early_seconds = layer_7_averages.get(piece_id, np.nan)
        mad_late_seconds = layer_8_averages.get(piece_id, np.nan)

        # Trial-level row++
        trial_rows.append({
            "subject_id": subject_id,
            "trial_number": trial_number,
            "piece_id": piece_id,
            "genre": genre,
            "overall_rating": overall_rating,
            "mad_early_seconds": mad_early_seconds,
            "mad_late_seconds": mad_late_seconds
        })

        # Time-series rows
        for i in range(3601):
            timeseries_rows.append({
                "subject_id": subject_id,
                "trial_number": trial_number,
                "piece_id": piece_id,
                "genre": genre,
                "time": dial_times[i],
                "sample_index": i,
                "dial_value": dial_vals[i],
                "overall_rating": overall_rating,
                "mad_early_seconds": mad_early_seconds,
                "mad_late_seconds": mad_late_seconds
            })

trial_df = pd.DataFrame(trial_rows)
timeseries_df = pd.DataFrame(timeseries_rows)

print("trial_df shape:", trial_df.shape)
print("timeseries_df shape:", timeseries_df.shape)

Found 27 pickle files
Loading: s01mi_aud_beh1_out_2026-02-25_16h27.19.437.pickle
{'expInfo': {'participant_number': '01', 'experimenter': 'Sophia S.', 'instructions': True, 'practice': True, 'screenNum': 1, 'fullScreen': True, 'screenName': 'TRACC_DellP2214H', 'date': '2026-02-25_16h27.19.437', 'expName': 'mi_aud_beh1'}, 'winVars': {'size': array([1920, 1080], dtype=int32), 'winType': 'pyglet', 'allowGUI': False, 'colorSpace': 'rgb', 'units': 'deg', 'useFBO': False, 'waitBlanking': True}, 'dial_calib_vals': {'cal_minimum': 165, 'cal_maximum': 199, 'cal_before_wrap': 3, 'cal_after_wrap': 0, 'cal_middle_wrap': 0, 'cal_offset_1': 91, 'cal_summed_ratings': 290, 'cal_Special_wrap': False}, 'stateDur': array([   5. , 1000. ,    5. , 1000. ,    3.5]), 'expStart': 395.6121900002472, 'trialInfo': {'trial': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32], 'startTime': [3.5098164998926222, 74.04398489976302, 146.5937486998737

NameError: name 'layer_7_averages' is not defined

In [ ]:
print("Subjects:", sorted(trial_df["subject_id"].unique()))
print("\nTrials per subject:")
print(trial_df.groupby("subject_id")["trial_number"].nunique())

print("\nMissing MAD values in trial_df:")
print(trial_df[["mad_early_seconds", "mad_late_seconds"]].isna().sum())

trial_out = OUTPUT_DIR / "behavior_trial_level.parquet"
timeseries_out = OUTPUT_DIR / "behavior_timeseries_level.parquet"

trial_df.to_parquet(trial_out, index=False)
timeseries_df.to_parquet(timeseries_out, index=False)

print(f"Saved trial-level data to: {trial_out}")
print(f"Saved timeseries-level data to: {timeseries_out}")

Subjects: ['01', '02', '03', '04', '05', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '24', '25', '27', '28', '29', '31']

Trials per subject:
subject_id
01    32
02    32
03    32
04    32
05    32
07    32
08    32
09    32
10    32
11    32
12    32
13    32
14    32
15    32
16    32
17    32
18    32
19    32
20    32
21    32
22    32
24    32
25    32
27    32
28    32
29    32
31    32
Name: trial_number, dtype: int64

Missing MAD values in trial_df:
mad_early_seconds    0
mad_late_seconds     0
dtype: int64
Saved trial-level data to: ../output/behavior_trial_level.parquet
Saved timeseries-level data to: ../output/behavior_timeseries_level.parquet


In [7]:
# ==========================
# 5) Subject-by-clip summary metrics
# ==========================

def compute_slope(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 2 or np.allclose(x, x[0]):
        return np.nan

    return np.polyfit(x, y, 1)[0]

group_cols = [
    "subject_id", "trial_number", "piece_id", "genre", "overall_rating",
    "mad_early_seconds", "mad_late_seconds"
]

subject_clip_summary = (
    timeseries_df
    .groupby(group_cols, as_index=False)
    .agg(
        mean_continuous_rating=("dial_value", "mean"),
        continuous_sd=("dial_value", lambda x: x.std(ddof=1)),
        continuous_slope=("dial_value", lambda y: compute_slope(
            timeseries_df.loc[y.index, "time"], y
        ))
    )
)

print("subject_clip_summary shape:", subject_clip_summary.shape)
display(subject_clip_summary.head())

subject_clip_summary shape: (864, 10)


,subject_id,trial_number,piece_id,genre,overall_rating,mad_early_seconds,mad_late_seconds,mean_continuous_rating,continuous_sd,continuous_slope
0,01,1,classical_11,classical,0.958621,0.282571,0.293700,0.880241,0.132011,0.006323
1,01,2,classical_12,classical,0.548276,0.282331,0.296777,0.554508,0.043877,0.002223
2,01,3,classical_16,classical,0.682759,0.286818,0.298212,0.660784,0.105568,0.005577
3,01,4,classical_5,classical,0.637931,0.283910,0.296515,0.629012,0.085981,0.000206
4,01,5,classical_15,classical,0.903448,0.281320,0.295139,0.812561,0.102621,0.000920


In [8]:
# ==========================
# 6) Clip-level summary metrics
# ==========================

clip_summary = (
    subject_clip_summary
    .groupby(["piece_id", "genre", "mad_early_seconds", "mad_late_seconds"], as_index=False)
    .agg(
        mean_overall_rating=("overall_rating", "mean"),
        sd_overall_rating=("overall_rating", "std"),
        mean_continuous_rating=("mean_continuous_rating", "mean"),
        sd_continuous_rating=("mean_continuous_rating", "std"),
        mean_continuous_variability=("continuous_sd", "mean"),
        mean_continuous_slope=("continuous_slope", "mean"),
        n_subjects=("subject_id", "nunique")
    )
)

print("clip_summary shape:", clip_summary.shape)
display(clip_summary.head())

clip_summary shape: (32, 11)


,piece_id,genre,mad_early_seconds,mad_late_seconds,mean_overall_rating,sd_overall_rating,mean_continuous_rating,sd_continuous_rating,mean_continuous_variability,mean_continuous_slope,n_subjects
0,classical_1,classical,0.285978,0.298822,0.725764,0.207362,0.696818,0.170989,0.094617,0.002906,27
1,classical_10,classical,0.281808,0.292032,0.624516,0.207599,0.630550,0.139628,0.110269,0.002056,27
2,classical_11,classical,0.282571,0.293700,0.650607,0.216631,0.622951,0.172071,0.105447,0.002923,27
3,classical_12,classical,0.282331,0.296777,0.668718,0.160995,0.646289,0.142685,0.093404,0.003149,27
4,classical_13,classical,0.283588,0.296210,0.654798,0.178246,0.661555,0.128072,0.102317,0.003263,27


In [9]:
# ==========================
# 7) Overall rating disagreement (z-scored within subject)
# ==========================

trial_df_z = trial_df.copy()

trial_df_z["overall_rating_z"] = (
    trial_df_z.groupby("subject_id")["overall_rating"]
    .transform(lambda x: (x - x.mean()) / x.std(ddof=1))
)

overall_disagreement = (
    trial_df_z
    .groupby(["piece_id", "genre", "mad_early_seconds", "mad_late_seconds"], as_index=False)
    .agg(
        overall_disagreement_sd=("overall_rating_z", "std"),
        n_subjects=("subject_id", "nunique")
    )
)

print("overall_disagreement shape:", overall_disagreement.shape)
display(overall_disagreement.head())

overall_disagreement shape: (32, 6)


,piece_id,genre,mad_early_seconds,mad_late_seconds,overall_disagreement_sd,n_subjects
0,classical_1,classical,0.285978,0.298822,0.922368,27
1,classical_10,classical,0.281808,0.292032,0.697041,27
2,classical_11,classical,0.282571,0.293700,0.809554,27
3,classical_12,classical,0.282331,0.296777,0.656225,27
4,classical_13,classical,0.283588,0.296210,0.673446,27


In [10]:
# ==========================
# 8) Continuous disagreement metrics
# ==========================

continuous_disagreement_time = (
    timeseries_df
    .groupby(
        ["piece_id", "genre", "mad_early_seconds", "mad_late_seconds", "sample_index", "time"],
        as_index=False
    )
    .agg(
        variance_across_participants=("dial_value", "var")
    )
)

summary_rows = []

for (piece_id, genre, mad_early, mad_late), g in continuous_disagreement_time.groupby(
    ["piece_id", "genre", "mad_early_seconds", "mad_late_seconds"]
):
    g = g.sort_values("time")

    summary_rows.append({
        "piece_id": piece_id,
        "genre": genre,
        "mad_early_seconds": mad_early,
        "mad_late_seconds": mad_late,
        "mean_continuous_disagreement": g["variance_across_participants"].mean(),
        "time_to_maximal_convergence": g.loc[
            g["variance_across_participants"].idxmin(), "time"
        ]
    })

continuous_disagreement_summary = pd.DataFrame(summary_rows)

print("continuous_disagreement_summary shape:", continuous_disagreement_summary.shape)
display(continuous_disagreement_summary.head())

continuous_disagreement_summary shape: (32, 6)


,piece_id,genre,mad_early_seconds,mad_late_seconds,mean_continuous_disagreement,time_to_maximal_convergence
0,classical_1,classical,0.285978,0.298822,0.009327,9.507750
1,classical_10,classical,0.281808,0.292032,0.025096,30.853875
2,classical_11,classical,0.282571,0.293700,0.060637,56.605937
3,classical_12,classical,0.282331,0.296777,0.010118,8.791806
4,classical_13,classical,0.283588,0.296210,0.101480,4.770086


In [11]:
# ==========================
# 9) Final clip-level analysis table
# ==========================

clip_analysis_df = (
    clip_summary
    .merge(
        overall_disagreement[["piece_id", "overall_disagreement_sd"]],
        on="piece_id",
        how="left"
    )
    .merge(
        continuous_disagreement_summary[[
            "piece_id",
            "mean_continuous_disagreement",
            "time_to_maximal_convergence"
        ]],
        on="piece_id",
        how="left"
    )
)

print("clip_analysis_df shape:", clip_analysis_df.shape)
display(clip_analysis_df.head())

clip_analysis_df shape: (32, 14)


,piece_id,genre,mad_early_seconds,mad_late_seconds,mean_overall_rating,sd_overall_rating,mean_continuous_rating,sd_continuous_rating,mean_continuous_variability,mean_continuous_slope,n_subjects,overall_disagreement_sd,mean_continuous_disagreement,time_to_maximal_convergence
0,classical_1,classical,0.285978,0.298822,0.725764,0.207362,0.696818,0.170989,0.094617,0.002906,27,0.922368,0.009327,9.507750
1,classical_10,classical,0.281808,0.292032,0.624516,0.207599,0.630550,0.139628,0.110269,0.002056,27,0.697041,0.025096,30.853875
2,classical_11,classical,0.282571,0.293700,0.650607,0.216631,0.622951,0.172071,0.105447,0.002923,27,0.809554,0.060637,56.605937
3,classical_12,classical,0.282331,0.296777,0.668718,0.160995,0.646289,0.142685,0.093404,0.003149,27,0.656225,0.010118,8.791806
4,classical_13,classical,0.283588,0.296210,0.654798,0.178246,0.661555,0.128072,0.102317,0.003263,27,0.673446,0.101480,4.770086
